### Load Data

In [1]:
from pathlib import Path
import pandas as pd

# Costs
costs_2024 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\costs\costs_2024.parquet")
costs = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\costs\costs.parquet")

# Prices
prices_2024 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\prices\prices_2024.parquet")
prices = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\prices\prices.parquet")

# Simulation daily
# simulation_daily_2024 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_daily\sim_daily_2024.parquet")
# simulation_daily_2023 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_daily\sim_daily_2023.parquet")
# simulation_daily_2022 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_daily\sim_daily_2022.parquet")
# simulation_daily_2021 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_daily\sim_daily_2021.parquet")
# simulation_daily_2020 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_daily\sim_daily_2020.parquet")

# Simulation monthly
# simulation_monthly_2024 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_monthly\sim_monthly_2024.parquet")
# simulation_monthly_2023 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_monthly\sim_monthly_2023.parquet")
# simulation_monthly_2022 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_monthly\sim_monthly_2022.parquet")
# simulation_monthly_2021 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_monthly\sim_monthly_2021.parquet")
# simulation_monthly_2020 = pd.read_parquet(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data\sim_monthly\sim_monthly_2020.parquet")

In [2]:
print("PRICE DATAFRAME INFO")
print(prices.info())

print("COST DATAFRAME INFO")
print(costs.info())

# print("SIM DAILY DATAFRAME INFO")
# print(simulation_daily_2024.info())

# print("SIM MONTHLY DATAFRAME INFO")
# print(simulation_monthly_2024.info())

PRICE DATAFRAME INFO
<class 'pandas.DataFrame'>
RangeIndex: 453167 entries, 0 to 453166
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   EID            453167 non-null  int16         
 1   DATETIME       453167 non-null  datetime64[ns]
 2   PEAKID         453167 non-null  int8          
 3   PRICEREALIZED  453167 non-null  float32       
dtypes: datetime64[ns](1), float32(1), int16(1), int8(1)
memory usage: 6.5 MB
None
COST DATAFRAME INFO
<class 'pandas.DataFrame'>
RangeIndex: 9092 entries, 0 to 9091
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   EID     9092 non-null   int16  
 1   MONTH   9092 non-null   str    
 2   PEAKID  9092 non-null   int8   
 3   C       9092 non-null   float32
dtypes: float32(1), int16(1), int8(1), str(1)
memory usage: 195.5 KB
None


### Use Polars for Simulation Data
* Using pandas crashed session

In [31]:
import polars as pl
from pathlib import Path

base_dir = Path(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data")

# Lazy scan keeps memory low and pushes filters to parquet read
daily_paths = [
    base_dir / "sim_daily" / "sim_daily_2024.parquet",
    base_dir / "sim_daily" / "sim_daily_2023.parquet",
    base_dir / "sim_daily" / "sim_daily_2022.parquet",
    base_dir / "sim_daily" / "sim_daily_2021.parquet",
    base_dir / "sim_daily" / "sim_daily_2020.parquet",
]

monthly_paths = [
    base_dir / "sim_monthly" / "sim_monthly_2024.parquet",
    base_dir / "sim_monthly" / "sim_monthly_2023.parquet",
    base_dir / "sim_monthly" / "sim_monthly_2022.parquet",
    base_dir / "sim_monthly" / "sim_monthly_2021.parquet",
    base_dir / "sim_monthly" / "sim_monthly_2020.parquet",
]

daily_lf = pl.scan_parquet(daily_paths)
monthly_lf = pl.scan_parquet(monthly_paths)

# Inspect schema without reading full data
daily_schema = daily_lf.schema
monthly_schema = monthly_lf.schema

# Small preview (collects only a few rows)
# daily_preview = daily_lf.limit(50).collect()
monthly_preview = monthly_lf.limit(3000000).collect()

C:\Users\akobe\AppData\Local\Temp\ipykernel_16428\983787611.py:27: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  daily_schema = daily_lf.schema
C:\Users\akobe\AppData\Local\Temp\ipykernel_16428\983787611.py:28: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  monthly_schema = monthly_lf.schema


In [ ]:
import polars as pl
from pathlib import Path

base_dir = Path(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data")
path_2023 = base_dir / "sim_monthly" / "sim_monthly_2023.parquet"

# Count unique scenarios in 2023
n_scenarios = (
    pl.scan_parquet(path_2023)
      .select(pl.col("SCENARIOID"))
      .unique()
      .collect()
      .height
)
print(f"Unique SCENARIOID in 2023 sim_monthly: {n_scenarios}")

# Mean feature profile per scenario (sample columns)
scenario_means = (
    pl.scan_parquet(path_2023)
      .group_by("SCENARIOID")
      .agg([
          pl.col("PSM").mean().alias("psm_mean"),
          pl.col("ACTIVATIONLEVEL").mean().alias("activation_mean"),
          pl.col("WINDIMPACT").mean().alias("wind_mean"),
          pl.col("SOLARIMPACT").mean().alias("solar_mean"),
          pl.col("HYDROIMPACT").mean().alias("hydro_mean"),
          pl.col("NONRENEWBALIMPACT").mean().alias("nonrenew_mean"),
          pl.col("EXTERNALIMPACT").mean().alias("external_mean"),
          pl.col("LOADIMPACT").mean().alias("load_mean"),
          pl.col("TRANSMISSIONOUTAGEIMPACT").mean().alias("transmission_mean"),
      ])
      .sort("SCENARIOID")
      .collect()
)

scenario_means.head(10)

Unique SCENARIOID in 2023 sim_monthly: 3


SCENARIOID,psm_mean,activation_mean,wind_mean,solar_mean,hydro_mean,nonrenew_mean,external_mean,load_mean,transmission_mean
i8,f32,f32,f32,f32,f32,f32,f32,f32,f32
1,-0.239355,26.632219,13.330543,0.007553,0.667263,8.394155,0.16083,0.719802,0.62341
2,-0.353354,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.41525,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
import polars as pl
from pathlib import Path

base_dir = Path(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data")
monthly_paths = [
    base_dir / "sim_monthly" / "sim_monthly_2024.parquet",
    base_dir / "sim_monthly" / "sim_monthly_2023.parquet",
    base_dir / "sim_monthly" / "sim_monthly_2022.parquet",
    base_dir / "sim_monthly" / "sim_monthly_2021.parquet",
    base_dir / "sim_monthly" / "sim_monthly_2020.parquet",
]

# Aggregate sim_monthly by (EID, PEAKID, MONTH) with mean features
sim_monthly_agg = (
    pl.scan_parquet(monthly_paths)
      .with_columns(
          pl.col("DATETIME").dt.strftime("%Y-%m").alias("MONTH")
      )
      .group_by(["EID", "PEAKID", "MONTH"])
      .agg([
          pl.col("ACTIVATIONLEVEL").mean().alias("activation_mean"),
          pl.col("WINDIMPACT").mean().alias("wind_mean"),
          pl.col("SOLARIMPACT").mean().alias("solar_mean"),
          pl.col("HYDROIMPACT").mean().alias("hydro_mean"),
          pl.col("NONRENEWBALIMPACT").mean().alias("nonrenew_mean"),
          pl.col("EXTERNALIMPACT").mean().alias("external_mean"),
          pl.col("LOADIMPACT").mean().alias("load_mean"),
          pl.col("TRANSMISSIONOUTAGEIMPACT").mean().alias("transmission_mean"),
          pl.col("PSM").mean().alias("psm_mean"),
      ])
      .collect()
)

# Optional: convert to pandas for joining with costs
sim_monthly_agg_pd = sim_monthly_agg.to_pandas()

sim_monthly_agg_pd.head()

In [ ]:
sim_monthly_agg_pd

In [7]:
import polars as pl
from pathlib import Path

base_dir = Path(r"C:\Users\akobe\OneDrive\Documents\mag-hec\data")

# How many unique EIDs are in sim_monthly for a single month?
lf = pl.scan_parquet([
    base_dir / "sim_monthly" / "sim_monthly_2022.parquet"
])

result = (lf
    .filter(
        (pl.col("DATETIME") >= pl.lit("2022-03-01").str.to_datetime("%Y-%m-%d")) &
        (pl.col("DATETIME") <  pl.lit("2022-04-01").str.to_datetime("%Y-%m-%d"))
    )
    .select(["EID", "PEAKID"])
    .unique()
    .collect()
)

print(f"Unique (EID, PEAKID) in sim_monthly for 2022-03: {len(result):,}")
print(f"Our truth table has ~729 rows for that month")
print(f"Ratio: {len(result) / 729:.0f}x more candidates in sim than in truth")

Unique (EID, PEAKID) in sim_monthly for 2022-03: 12,038
Our truth table has ~729 rows for that month
Ratio: 17x more candidates in sim than in truth
